<a href="https://colab.research.google.com/github/Darioldeveloper-dev/CSC4110-G01/blob/main/G01_U2_1_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 4110 Machine Learning
## Group G01: Student Performance Factors
### Task 2.1: Data Inspection and Cleaning

- **Owner:** Dariol Temko
- **Registration number:** COM/0048/23
- **Module:** 3

**Contents**
1. Dataset loading
2. Data inspection
3. Cleaning (missing values, capping, range checks, IQR)
4. Data types and saving `stage1_students.csv`
5. Median imputation by hand (exam skill)
6. Written answer

In [43]:
# Mount Google Drive and import pandas
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

# Load the raw dataset from the project data folder
df = pd.read_csv("/content/drive/MyDrive/CSC4110_G01/data/StudentPerformanceFactors.csv")

# Keep an untouched backup copy of the raw data
df_raw = df.copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
# Column names, data types and non-blank counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6607 entries, 0 to 6606
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   Hours_Studied               6607 non-null   int64 
 1   Attendance                  6607 non-null   int64 
 2   Parental_Involvement        6607 non-null   object
 3   Access_to_Resources         6607 non-null   object
 4   Extracurricular_Activities  6607 non-null   object
 5   Sleep_Hours                 6607 non-null   int64 
 6   Previous_Scores             6607 non-null   int64 
 7   Motivation_Level            6607 non-null   object
 8   Internet_Access             6607 non-null   object
 9   Tutoring_Sessions           6607 non-null   int64 
 10  Family_Income               6607 non-null   object
 11  Teacher_Quality             6529 non-null   object
 12  School_Type                 6607 non-null   object
 13  Peer_Influence              6607 non-null   obje

In [45]:
# Missing values per column, and number of fully duplicated rows
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Education_Level      90
Distance_from_Home            67
Gender                         0
Exam_Score                     0
dtype: int64
Duplicate rows: 0


In [46]:
# Summary statistics for the numeric columns
df.describe()


,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [47]:
# Value counts of the column the classification target will be built from
(df['Exam_Score'] >= 70).value_counts()

,count
Exam_Score,
False,4982
True,1625


### Data Quality and Target Balance

**Duplicates:** The dataset contains 0 duplicate rows, indicating that no repeated records were identified.

**Missing Values:** Three columns contain missing values:
- `Teacher_Quality`: 78 missing values.
- `Parental_Education_Level`: 90 missing values.
- `Distance_from_Home`: 67 missing values.

This gives a total of 235 missing values, with each column having approximately 1% missing data.

**Target Balance:** Based on the proposed classification threshold of 70 marks:
- **True (70 or above):** 1,625 students (24.6%).
- **False (below 70):** 4,982 students (75.4%).

Therefore, approximately 24.6% of students fall into the potential mentorship scholarship nomination group, while 75.4% fall below the threshold. This indicates a class imbalance that should be considered when developing and evaluating the classification model.

In [48]:
# Start the cleaning copy; df_raw stays untouched
df_stage1 = df_raw.copy()

# Category columns that contain blanks
missing_cols = ['Teacher_Quality', 'Parental_Education_Level', 'Distance_from_Home']

# For each column: count blanks, find the most common value, then fill the blanks with it
for col in missing_cols:
    n_missing = df_stage1[col].isnull().sum()
    mode_value = df_stage1[col].mode()[0]
    df_stage1[col] = df_stage1[col].fillna(mode_value)
    print(f"{col}: filled {n_missing} blanks with '{mode_value}'")

# Confirm that no blanks remain anywhere
print("Blanks left:", df_stage1.isnull().sum().sum())

Teacher_Quality: filled 78 blanks with 'Medium'
Parental_Education_Level: filled 90 blanks with 'High School'
Distance_from_Home: filled 67 blanks with 'Near'
Blanks left: 0


Deleting the blank cells looses 235 students

In [49]:
# Settings for the allowed range of Exam_Score
outlier_col = "Exam_Score"
lower = 0
upper = 100

# Show the rows that break the rule, before changing anything
bad = df_stage1[(df_stage1[outlier_col] < lower) | (df_stage1[outlier_col] > upper)]
print("Values outside", lower, "to", upper, ":", len(bad))
print(bad[[outlier_col]])

# Cap every value into the allowed range
df_stage1[outlier_col] = df_stage1[outlier_col].clip(lower=lower, upper=upper)

# Confirm the new maximum
print("New max:", df_stage1[outlier_col].max())

Values outside 0 to 100 : 1
      Exam_Score
1525         101
New max: 100


### Data Cleaning and Preprocessing Decisions

**Missing Value Imputation:** All 235 missing values across the three categorical columns were filled using the mode (most frequent category) because the mode suits category columns because words have no mean or middle value; filling keeps every row, and since only about 1% of each column was blank, the effect on the distribution is small
- `Teacher_Quality`: Filled 78 missing values with `Medium` (mode suits categorical features).
- `Parental_Education_Level`: Filled 90 missing values with `High School` (mode suits categorical features).
- `Distance_from_Home`: Filled 67 missing values with `Near` (mode suits categorical features).

**Outlier Handling:** One invalid score was identified and corrected in the target variable:
- `Exam_Score`: Capped 1 record from `101` to `100` (impossible mark exceeding the maximum exam score).

Therefore,
missing values and invalid entries have been resolved, leaving a complete and valid dataset of 6,607 records ready for feature encoding and model development.

In [50]:
# Check Attendance is within 0-100 and Sleep_Hours within 0-24
bad_attendance = df_stage1[(df_stage1['Attendance'] < 0) | (df_stage1['Attendance'] > 100)]
bad_sleep = df_stage1[(df_stage1['Sleep_Hours'] < 0) | (df_stage1['Sleep_Hours'] > 24)]
print("Attendance outside 0-100:", len(bad_attendance))
print("Sleep_Hours outside 0-24:", len(bad_sleep))

Attendance outside 0-100: 0
Sleep_Hours outside 0-24: 0


In [51]:
# IQR rule: find values unusually far from the middle 50% of the data
q1 = df_stage1['Hours_Studied'].quantile(0.25)
q3 = df_stage1['Hours_Studied'].quantile(0.75)
iqr = q3 - q1
low_fence = q1 - 1.5 * iqr
high_fence = q3 + 1.5 * iqr

# Select the flagged rows and report them
flagged = df_stage1[(df_stage1['Hours_Studied'] < low_fence) | (df_stage1['Hours_Studied'] > high_fence)]
print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Fences:", low_fence, "to", high_fence)
print("Flagged values:", len(flagged))
print(flagged['Hours_Studied'].describe())

Q1: 16.0 Q3: 24.0 IQR: 8.0
Fences: 4.0 to 36.0
Flagged values: 43
count    43.000000
mean     20.906977
std      18.318367
min       1.000000
25%       3.000000
50%      37.000000
75%      38.000000
max      44.000000
Name: Hours_Studied, dtype: float64


**Outlier Retention (`Hours_Studied`):** Using the IQR rule ($Q_1 = 16$, $Q_3 = 24$, $\text{IQR} = 8$), 43 records fall outside the 4 to 36-hour fences, ranging from 1 to 44 hours per week:
- `Hours_Studied`: Retained all 43 flagged records without modification.

Although statistically uncommon, studying 1 hour or 44 hours per week is completely realistic for different types of students. Because the IQR rule only highlights extreme values rather than invalid data, dropping these 43 out of 6607 students would remove genuine variation and valuable information from the dataset.

In [52]:
# Look for numbers stored as text
print(df_stage1.dtypes)
text_numbers = []
for col in df_stage1.select_dtypes(include='object').columns:
    # Columns that convert fully to numbers are numbers stored as text
    if pd.to_numeric(df_stage1[col], errors='coerce').notnull().all():
        text_numbers.append(col)
print("Numbers stored as text:", text_numbers)

Hours_Studied                  int64
Attendance                     int64
Parental_Involvement          object
Access_to_Resources           object
Extracurricular_Activities    object
Sleep_Hours                    int64
Previous_Scores                int64
Motivation_Level              object
Internet_Access               object
Tutoring_Sessions              int64
Family_Income                 object
Teacher_Quality               object
School_Type                   object
Peer_Influence                object
Physical_Activity              int64
Learning_Disabilities         object
Parental_Education_Level      object
Distance_from_Home            object
Gender                        object
Exam_Score                     int64
dtype: object
Numbers stored as text: []


In [53]:
# Save the cleaned table for Task 2.2
df_stage1.to_csv("/content/drive/MyDrive/CSC4110_G01/data/stage1_students.csv", index=False)

# Read it back to prove it saved correctly
check = pd.read_csv("/content/drive/MyDrive/CSC4110_G01/data/stage1_students.csv")
print(check.shape, "blanks:", check.isnull().sum().sum(), "max score:", check['Exam_Score'].max())

(6607, 20) blanks: 0 max score: 100


In [54]:
import numpy as np

# Take the first eight Hours_Studied values from the raw data
eight = df_raw['Hours_Studied'].head(8)
print("Original:", eight.tolist())

# Copy as decimals, because whole-number columns cannot hold a missing value cleanly
eight_missing = eight.astype(float).copy()

# Set the third value (position 2, counting from 0) to missing
eight_missing.iloc[2] = np.nan
print("With one missing:", eight_missing.tolist())

Original: [23, 19, 24, 29, 19, 19, 29, 25]
With one missing: [23.0, 19.0, nan, 29.0, 19.0, 19.0, 29.0, 25.0]


### Median Calculation (Step-by-Step Verification)

**Non-Blank Values:** After removing the blank (`nan`) entry from the 8 observations, seven valid values remain:
- `[23, 19, 29, 19, 19, 29, 25]`

**Sorted Order:** Arranging the seven remaining values from smallest to largest gives:
- `[19, 19, 19, 23, 25, 29, 29]`

**Median Identification:** With $n = 7$ observations, the middle position is given by $\frac{7 + 1}{2} = 4$:
- **Middle Position:** 4th value in the sorted list.
- **Median Value:** `23`

Therefore, after excluding the missing value and ordering the remaining seven observations, the 4th value (`23`) represents the median of the sample.

In [55]:
# Fill the missing value with the median of the other values
filled = eight_missing.fillna(eight_missing.median())
print("After filling:", filled.tolist())

After filling: [23.0, 19.0, 23.0, 29.0, 19.0, 19.0, 29.0, 25.0]


### Median Imputation Code Explanation

**Median Calculation (`eight_missing.median()`):** This method computes the middle value of the series, returning `23.0`:
- By default (`skipna=True`), pandas automatically ignores the blank (`NaN`) entry and calculates the median using only the seven valid numbers.

**Missing Value Replacement (`fillna(...)`):** This method searches the series for any missing (`NaN`) positions and replaces them with the value passed inside the parentheses (`23.0`), while leaving all existing non-blank values unchanged.

**Comparison with Hand Calculation:** The resulting list (`[23.0, 19.0, 23.0, 29.0, 19.0, 19.0, 29.0, 25.0]`) matches the manual calculation:
- The 3rd position (previously `NaN`) is replaced with `23.0`, confirming that pandas performs the exact same exclusion and sorting steps under the hood.

In [56]:
# Self-check of the graded evidence
assert df_raw.shape == (6607, 20)
assert df_raw.isnull().sum().sum() == 235          # raw data still has its blanks
assert df_raw['Exam_Score'].max() == 101           # raw data still has the impossible value
assert df_stage1.isnull().sum().sum() == 0         # cleaned data has no blanks
assert df_stage1['Exam_Score'].between(lower, upper).all()
print("All checks passed")

All checks passed


### Summary of Data Cleaning Decisions

**Cleaning Overview:** Every preprocessing decision applied to the dataset (6,607 rows and 20 columns) is documented below alongside the number of affected records and its justification:

| Decision | Affected | Reason |
| :--- | :--- | :--- |
| Rows removed | 0 | No duplicate rows were found in the dataset |
| Columns dropped | 0 | All 20 features are relevant and were retained |
| Mode fills (three columns) | 78 + 90 + 67 = 235 | Mode is the most appropriate measure for categorical features |
| `Exam_Score` capped | 1 | Capped 101 to 100 to correct an impossible exam mark |
| Range checks | 0 | All remaining numerical features fell within valid domain limits |
| IQR flags kept | 43 | Extreme study hours (1–44 hrs/week) are realistic and retain real variation |
| Types changed | 0 | All 20 columns already had appropriate data types (`int64` and `object`) |

Therefore, all data quality issues have been systematically addressed while preserving 100% of the valid student records for downstream modeling.